# 02 — Aurora + Apache AGE Knowledge Graph

Replaces Neptune Analytics with **Aurora Serverless v2 + Apache AGE**.

| Concern | Solution |
|---|---|
| Graph queries | Cypher via `ag_catalog.cypher()` over RDS Data API |
| Vector search | pgvector on the same cluster (see notebook 04) |
| Idle cost | ~$0.06/hr (0.5 ACU min, shared with KB) |
| Access | RDS Data API — no VPC, no bastion |

Run this notebook once after deploying `OrgIQ-Aurora` to initialise the graph.

## Step 1: Load Configuration & Build RDS Client

In [ ]:
import boto3
import json
from pathlib import Path

config_path = Path("config.json")
config = json.loads(config_path.read_text()) if config_path.exists() else {}

REGION             = config.get("REGION", "us-east-1")
PREFIX             = config.get("PREFIX", "orgiq")
AURORA_CLUSTER_ARN = config.get("AURORA_CLUSTER_ARN", "")
AURORA_SECRET_ARN  = config.get("AURORA_SECRET_ARN",  "")
DB_NAME            = config.get("AURORA_DB_NAME",   "orgiq")
GRAPH_NAME         = config.get("AURORA_GRAPH_NAME","orgiq_graph")

if not AURORA_CLUSTER_ARN:
    raise SystemExit(
        "✗ AURORA_CLUSTER_ARN missing in config.json.\n"
        "  Deploy the Aurora stack first: cd infra && cdk deploy OrgIQ-Aurora\n"
        "  Then paste the ClusterArn output into config.json."
    )

rds = boto3.client("rds-data", region_name=REGION)

# ── Helpers ──────────────────────────────────────────────────────────

def _exec(sql: str) -> dict:
    return rds.execute_statement(
        resourceArn=AURORA_CLUSTER_ARN,
        secretArn=AURORA_SECRET_ARN,
        database=DB_NAME,
        sql=sql,
    )

def _parse_agtype(v: str):
    """Strip AGE type annotations and parse as JSON."""
    try:
        return json.loads(v)
    except Exception:
        return v

def cypher(query: str, columns: str = "result ag_catalog.agtype") -> list[dict]:
    """
    Execute a Cypher query via Apache AGE.

    columns: the AS clause, e.g. 'name ag_catalog.agtype, score ag_catalog.agtype'
    Each column is returned as a parsed Python value.
    """
    sql = (
        f"SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', $$ {query} $$)"
        f" AS ({columns})"
    )
    resp = _exec(sql)
    col_names = [c.strip().split()[0] for c in columns.split(",")]
    rows = []
    for record in resp.get("records", []):
        row = {}
        for col_name, field in zip(col_names, record):
            raw = (field.get("stringValue")
                   or field.get("longValue")
                   or field.get("doubleValue")
                   or None)
            row[col_name] = _parse_agtype(raw) if isinstance(raw, str) else raw
        rows.append(row)
    return rows

print(f"✓ RDS Data API client ready")
print(f"  Cluster : {AURORA_CLUSTER_ARN.split(':cluster:')[-1]}")
print(f"  Database: {DB_NAME}")
print(f"  Graph   : {GRAPH_NAME}")

## Step 2: Initialise Apache AGE Extension and Graph

Idempotent — safe to re-run. Sets the database-level `search_path` so AGE functions are found in every subsequent connection without needing `SET search_path` per call.

In [ ]:
# 1. Install extensions (idempotent)
for sql in [
    "CREATE EXTENSION IF NOT EXISTS age",
    "CREATE EXTENSION IF NOT EXISTS vector",
]:
    try:
        _exec(sql)
        print(f"✓ {sql}")
    except Exception as e:
        print(f"  {sql[:50]}… → {e}")

# 2. Set search_path permanently at database level so every RDS Data API
#    connection finds ag_catalog without a per-session SET command.
try:
    _exec(
        "ALTER DATABASE orgiq "
        "SET search_path TO ag_catalog, public"
    )
    print("✓ search_path set on database (persistent)")
except Exception as e:
    print(f"  ALTER DATABASE search_path → {e}")

# 3. Create the AGE graph (idempotent)
try:
    _exec(f"SELECT ag_catalog.create_graph('{GRAPH_NAME}')")
    print(f"✓ Graph '{GRAPH_NAME}' created")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"✓ Graph '{GRAPH_NAME}' already exists")
    else:
        raise

## Step 3: Verify Graph Schema

**Node labels:** `Person`, `Team`, `Skill`, `Project`, `Document`  
**Relationship types:** `KNOWS_ABOUT`, `MEMBER_OF`, `WORKED_ON`, `AUTHORED`  

AGE enforces idempotency via `MERGE` (no `CREATE CONSTRAINT` syntax yet).

In [ ]:
for label in ["Person", "Team", "Skill"]:
    rows = cypher(
        f"MATCH (n:{label}) RETURN count(n) AS cnt",
        columns="cnt ag_catalog.agtype",
    )
    count = rows[0]["cnt"] if rows else 0
    print(f"  {label:10s}: {count} nodes")

## Step 4: Load Sample People

In [ ]:
people = [
    {"id": "alice",  "name": "Alice Chen",    "email": "alice@example.com",  "title": "Staff Engineer",     "team": "Platform"},
    {"id": "bob",    "name": "Bob Smith",     "email": "bob@example.com",    "title": "Senior Engineer",    "team": "Platform"},
    {"id": "carol",  "name": "Carol Johnson", "email": "carol@example.com",  "title": "Engineering Manager","team": "Platform"},
    {"id": "dave",   "name": "Dave Lee",      "email": "dave@example.com",   "title": "Senior Engineer",    "team": "Data"},
    {"id": "eve",    "name": "Eve Martinez",  "email": "eve@example.com",    "title": "Staff Engineer",     "team": "Data"},
    {"id": "frank",  "name": "Frank Wilson",  "email": "frank@example.com",  "title": "Engineering Manager","team": "Data"},
    {"id": "grace",  "name": "Grace Kim",     "email": "grace@example.com",  "title": "Principal Engineer", "team": "Security"},
    {"id": "henry",  "name": "Henry Brown",   "email": "henry@example.com",  "title": "Senior Engineer",    "team": "Security"},
]

for p in people:
    pid, name, email, title, team = p["id"], p["name"], p["email"], p["title"], p["team"]
    cypher(
        f"MERGE (n:Person {{id: '{pid}'}}) "
        f"SET n.name = '{name}', n.email = '{email}', "
        f"    n.title = '{title}', n.team = '{team}'",
        columns="n ag_catalog.agtype",
    )
    print(f"  MERGE Person: {name} ({team})")

print(f"\n✓ {len(people)} Person nodes seeded")

## Step 5: Load Skills and KNOWS_ABOUT Relationships

In [ ]:
skills = [
    "Kubernetes", "Python", "Rust", "Auth", "ML",
    "Security", "Data Engineering", "Postgres",
]

for skill in skills:
    cypher(f"MERGE (s:Skill {{name: '{skill}'}})", columns="s ag_catalog.agtype")
print(f"✓ {len(skills)} Skill nodes")

knows_about = [
    ("alice", "Kubernetes",       0.95),
    ("alice", "Python",           0.90),
    ("bob",   "Kubernetes",       0.75),
    ("bob",   "Rust",             0.80),
    ("carol", "Auth",             0.85),
    ("carol", "Kubernetes",       0.60),
    ("dave",  "ML",               0.90),
    ("dave",  "Python",           0.85),
    ("eve",   "Data Engineering", 0.92),
    ("eve",   "Postgres",         0.88),
    ("frank", "Data Engineering", 0.70),
    ("grace", "Security",         0.95),
    ("grace", "Auth",             0.90),
    ("henry", "Security",         0.80),
    ("henry", "Postgres",         0.75),
]

for pid, skill_name, strength in knows_about:
    cypher(
        f"MATCH (p:Person {{id: '{pid}'}}), (s:Skill {{name: '{skill_name}'}}) "
        f"MERGE (p)-[r:KNOWS_ABOUT]->(s) "
        f"SET r.strength = {strength}",
        columns="p ag_catalog.agtype",
    )

print(f"✓ {len(knows_about)} KNOWS_ABOUT relationships")

## Step 6: Load Teams and MEMBER_OF Relationships

In [ ]:
for team in ["Platform", "Data", "Security"]:
    cypher(f"MERGE (t:Team {{name: '{team}'}})", columns="t ag_catalog.agtype")

for p in people:
    pid, team = p["id"], p["team"]
    cypher(
        f"MATCH (person:Person {{id: '{pid}'}}), (t:Team {{name: '{team}'}}) "
        f"MERGE (person)-[:MEMBER_OF]->(t)",
        columns="person ag_catalog.agtype",
    )

print(f"✓ 3 Team nodes + {len(people)} MEMBER_OF relationships")

## Step 7: Query — Find Experts for a Skill

In [ ]:
topic = "Kubernetes"

rows = cypher(
    f"MATCH (p:Person)-[r:KNOWS_ABOUT]->(s:Skill {{name: '{topic}'}}) "
    f"RETURN p.name AS name, p.team AS team, r.strength AS strength "
    f"ORDER BY r.strength DESC",
    columns="name ag_catalog.agtype, team ag_catalog.agtype, strength ag_catalog.agtype",
)

print(f"Experts in {topic}:")
for row in rows:
    print(f"  {row['name']:<20} {row['team']:<12} strength={row['strength']}")

## Step 8: 2-Hop Traversal — Shared Skill Overlap

In [ ]:
rows = cypher(
    "MATCH (alice:Person {id: 'alice'})-[:KNOWS_ABOUT]->(s:Skill)"
    "<-[:KNOWS_ABOUT]-(colleague:Person)-[:MEMBER_OF]->(t:Team {name: 'Security'}) "
    "RETURN DISTINCT colleague.name AS colleague, s.name AS skill",
    columns="colleague ag_catalog.agtype, skill ag_catalog.agtype",
)

print("Security colleagues sharing skills with Alice:")
for row in rows:
    print(f"  {row['colleague']} — {row['skill']}")

## Step 9: Verify Node Counts

In [ ]:
for label in ["Person", "Team", "Skill", "Project", "Document"]:
    rows = cypher(
        f"MATCH (n:{label}) RETURN count(n) AS cnt",
        columns="cnt ag_catalog.agtype",
    )
    count = rows[0]["cnt"] if rows else 0
    print(f"  {label:<12}: {count}")

## Step 10: Save Configuration

In [ ]:
config["AURORA_CLUSTER_ARN"] = AURORA_CLUSTER_ARN
config["AURORA_SECRET_ARN"]  = AURORA_SECRET_ARN
config["AURORA_DB_NAME"]     = DB_NAME
config["AURORA_GRAPH_NAME"]  = GRAPH_NAME

config_path.write_text(json.dumps(config, indent=2))
print("✓ config.json updated")
print(f"  AURORA_CLUSTER_ARN : {AURORA_CLUSTER_ARN}")
print(f"  AURORA_GRAPH_NAME  : {GRAPH_NAME}")

## Summary

| Resource | Details |
|---|---|
| **Graph engine** | Aurora Serverless v2 + Apache AGE |
| **Query language** | Cypher via `ag_catalog.cypher()` |
| **Idle cost** | ~$0.06/hr shared with pgvector/KB (0.5 ACU min) |
| **search_path** | Set permanently on the `orgiq` database — no per-call `SET` needed |
| **Multi-column returns** | Declare each column in the `columns=` arg of `cypher()` |
| **Access method** | RDS Data API — no VPC or bastion required |

**Next:** Run notebook **04** to create the `bedrock_kb` pgvector table on the same cluster.